# Excluded Data Analysis

This notebook focuses on analyzing the two separate categories of excluded molecules:
1. **non_numeric_rows** - Molecules with missing or non-numeric IC50 values
2. **dropped_rows** - Molecules with IC50 values of 100 or 200 (manually removed outliers)

Both categories will receive predictions from the best model trained on the main dataset.

## Setup and Data Loading

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
np.random.seed(42)

print("="*80)
print("EXCLUDED DATA ANALYSIS & PREDICTIONS")
print("="*80)

# Change to workspace directory
os.chdir("/Users/nb/Documents/Tzu-qsar-generative-model")

# Import pipeline modules
from data_loader import (load_excel_sheets, apply_smiles_cleaning,
                         combine_and_deduplicate, filter_numeric_ic50,
                         standardize_smiles)
from descriptors import compute_descriptors
from model import train_and_select, predict_and_antilog
from rdkit import Chem

print("\n✓ Imports successful")

## 1. Load and Prepare Data

In [ ]:
print("\n[1/4] Loading and preparing data...")

# Load data
series_dfs = load_excel_sheets("TB Project QSAR.xlsx")
series_dfs = apply_smiles_cleaning(series_dfs)
df = combine_and_deduplicate(series_dfs)

# Remove duplicates
unique_df = df.drop_duplicates(subset=['Canonical_SMILES']).copy()

# SEPARATELY identify the two excluded categories
print("\n   Identifying excluded data categories...")

# Category 1: Non-numeric IC50 values
non_numeric_rows = unique_df[pd.to_numeric(unique_df["IC50 uM"], errors="coerce").isna()].copy()
print(f"   ✓ Non-numeric IC50 rows: {len(non_numeric_rows)}")

# Get numeric rows first
numeric_df_temp = unique_df[pd.to_numeric(unique_df["IC50 uM"], errors="coerce").notna()].copy()

# Category 2: Dropped rows (IC50 = 100, 200)
dropped_rows = numeric_df_temp[numeric_df_temp["IC50 uM"].isin([100, 200])].copy()
print(f"   ✓ Dropped rows (IC50=100,200): {len(dropped_rows)}")

# Get final numeric dataset (without dropped rows)
numeric_df = numeric_df_temp.drop(dropped_rows.index).copy()
print(f"   ✓ Final training set: {len(numeric_df)}")

# Log transform
numeric_df['transformed_IC50'] = np.log10(numeric_df['IC50 uM'] + 1e-8)

print(f"\n   Total molecules: {len(unique_df)}")
print(f"   → Used for training: {len(numeric_df)}")
print(f"   → Excluded (non-numeric): {len(non_numeric_rows)}")
print(f"   → Excluded (dropped): {len(dropped_rows)}")

## 2. Standardize SMILES for All Excluded Rows

In [ ]:
print("\n[2/4] Standardizing SMILES for excluded molecules...")

# Standardize SMILES for main dataset
numeric_df['cleanedMol'] = numeric_df['Canonical_SMILES'].apply(
    lambda x: standardize_smiles(x, verbose=False)
)
print(f"   ✓ Main dataset SMILES standardized: {len(numeric_df)}")

# Standardize SMILES for dropped rows
dropped_rows['cleanedMol'] = dropped_rows['Canonical_SMILES'].apply(
    lambda x: standardize_smiles(x, verbose=False)
)
print(f"   ✓ Dropped rows SMILES standardized: {len(dropped_rows)}")

# Standardize SMILES for non-numeric rows
non_numeric_rows['cleanedMol'] = non_numeric_rows['Canonical_SMILES'].apply(
    lambda x: standardize_smiles(x, verbose=False)
)
print(f"   ✓ Non-numeric rows SMILES standardized: {len(non_numeric_rows)}")

print("\n✅ All SMILES standardization complete")

## 3. Train Best Model and Generate Predictions

In [ ]:
print("\n[3/4] Training best model and generating predictions...")

# Prepare data splits
train_df = numeric_df[numeric_df["Series_Code"].isin(["A","B","D"])].reset_index(drop=True)
test_df = numeric_df[numeric_df["Series_Code"] == "C"].reset_index(drop=True)
predict_df = numeric_df[numeric_df["Series_Code"] == "E"].reset_index(drop=True)

# Compute descriptors for all datasets
X_train_raw = compute_descriptors(train_df["cleanedMol"].values)
X_test_raw = compute_descriptors(test_df["cleanedMol"].values)
X_dropped_raw = compute_descriptors(dropped_rows["cleanedMol"].values)
X_non_numeric_raw = compute_descriptors(non_numeric_rows["cleanedMol"].values)

y_train = train_df["transformed_IC50"].values
y_test = test_df["transformed_IC50"].values

# Train best model
trained_models, imputers, scalers, best_desc, best_model_name, results_df = \
    train_and_select(X_train_raw, y_train, X_test_raw, y_test)

print(f"\n   Best model: {best_model_name}")
print(f"   Descriptor type: {best_desc}")
print(f"   Best R² score: {results_df['R2'].max():.4f}")

# Get best model components
model_best = trained_models[(best_desc, best_model_name)]
imp_best = imputers[best_desc]
sc_best = scalers.get((best_desc, best_model_name), None)

# Transform data for best model
X_dropped_best = imp_best.transform(X_dropped_raw[best_desc])
X_non_numeric_best = imp_best.transform(X_non_numeric_raw[best_desc])

if sc_best:
    X_dropped_best = sc_best.transform(X_dropped_best)
    X_non_numeric_best = sc_best.transform(X_non_numeric_best)

# Generate predictions
pred_dropped_df = predict_and_antilog(model_best, X_dropped_best, dropped_rows)
pred_non_numeric_df = predict_and_antilog(model_best, X_non_numeric_best, non_numeric_rows)

print(f"\n   ✓ Predictions for {len(pred_dropped_df)} dropped rows")
print(f"   ✓ Predictions for {len(pred_non_numeric_df)} non-numeric IC50 rows")

print("\n✅ Model training and predictions complete")

## 4. Save and Display Results

In [ ]:
print("\n[4/4] Saving results to outputs/...\n")

# Create output directory
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

# Save dropped rows predictions
dropped_file = os.path.join(output_dir, "Dropped_rows_predictions.csv")
pred_dropped_df.to_csv(dropped_file, index=False)
print(f"✓ Saved: {dropped_file}")
print(f"  Rows: {len(pred_dropped_df)}")
print(f"  Columns: {len(pred_dropped_df.columns)}")
print(f"  Has cleanedMol: {'✓' if 'cleanedMol' in pred_dropped_df.columns else '✗'}")

# Save non-numeric rows predictions
non_numeric_file = os.path.join(output_dir, "Non_numeric_rows_predictions.csv")
pred_non_numeric_df.to_csv(non_numeric_file, index=False)
print(f"\n✓ Saved: {non_numeric_file}")
print(f"  Rows: {len(pred_non_numeric_df)}")
print(f"  Columns: {len(pred_non_numeric_df.columns)}")
print(f"  Has cleanedMol: {'✓' if 'cleanedMol' in pred_non_numeric_df.columns else '✗'}")

print("\n✅ All files saved successfully!")

## Results Summary

### Dropped Rows (IC50 = 100 or 200)
These are molecules that were manually removed from the training set as potential outliers.

In [ ]:
print("\n" + "="*80)
print("DROPPED ROWS (IC50 = 100, 200)")
print("="*80)
print(f"\nTotal: {len(pred_dropped_df)} molecules\n")
print(pred_dropped_df[['Canonical_SMILES', 'cleanedMol', 'IC50 uM', 'Predicted_IC50']].to_string())

### Non-Numeric IC50 Rows
These are molecules with missing or non-numeric IC50 values. The model provides predicted values for these.

In [ ]:
print("\n" + "="*80)
print("NON-NUMERIC IC50 ROWS")
print("="*80)
print(f"\nTotal: {len(pred_non_numeric_df)} molecules\n")
print(pred_non_numeric_df[['Canonical_SMILES', 'cleanedMol', 'IC50 uM', 'Predicted_IC50']].to_string())

## Comparison Table

In [ ]:
print("\n" + "="*80)
print("EXCLUDED DATA SUMMARY")
print("="*80)

summary_data = {
    'Category': ['Dropped Rows', 'Non-Numeric IC50', 'Total Excluded'],
    'Count': [len(pred_dropped_df), len(pred_non_numeric_df), len(pred_dropped_df) + len(pred_non_numeric_df)],
    'Original IC50': ['100 or 200', 'NaN/Missing', 'Mixed'],
    'Has cleanedMol': ['✓', '✓', '✓'],
    'Model Predictions': ['✓', '✓', '✓']
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("📊 OUTPUT FILES GENERATED:")
print("="*80)
print(f"\n1. Dropped_rows_predictions.csv")
print(f"   • {len(pred_dropped_df)} rows × {len(pred_dropped_df.columns)} columns")
print(f"   • Original IC50 values: 100 or 200 µM")
print(f"   • Predicted IC50: {pred_dropped_df['Predicted_IC50'].min():.2f} - {pred_dropped_df['Predicted_IC50'].max():.2f} µM")
print(f"   • Columns: cleanedMol, Predicted_IC50, and all metadata")

print(f"\n2. Non_numeric_rows_predictions.csv")
print(f"   • {len(pred_non_numeric_df)} rows × {len(pred_non_numeric_df.columns)} columns")
print(f"   • Original IC50 values: Missing/Non-numeric")
print(f"   • Predicted IC50: {pred_non_numeric_df['Predicted_IC50'].min():.2f} - {pred_non_numeric_df['Predicted_IC50'].max():.2f} µM")
print(f"   • Columns: cleanedMol, Predicted_IC50, and all metadata")

print(f"\n📌 Model Used: {best_model_name} with {best_desc} descriptors (R² = {results_df['R2'].max():.4f})")
print(f"📌 Location: {os.path.abspath(output_dir)}/")
print("\n" + "="*80)